# Chapter 24 — From score to an inspectable validation dashboard

Read `STUDY_CONTRACT.md` before running. All records, groups, sites, scores, and outcomes are synthetic. No EEG predictor, real OOD detector, clinical integration, or patient benefit is demonstrated. Outputs are intentionally empty; run and document your own observations.

Keep the helper beside this notebook. In Colab, upload `biomarker_validation.py` to the runtime working directory. The core needs Python 3.11+ and the standard library; plotting uses Matplotlib.


In [ ]:
import sys
from pathlib import Path
from pprint import pprint
assert sys.version_info >= (3, 11)
assert Path("biomarker_validation.py").is_file(), "Place/upload the helper beside the notebook."
import biomarker_validation as lab
print(lab.VERSION, sys.version)


## 1. Establish the study population and decision time

One synthetic person per row. Development A, calibration B, and external C are disjoint; all outcomes are complete. The candidate score is a deliberately distorted generator quantity, not a trained EEG model. G1/G2 are artificial groups, not biological claims.


In [ ]:
rows = lab.make_dataset()
cohorts = {name: [r for r in rows if r["cohort"] == name]
           for name in ("development", "calibration", "external")}
ids = {name: {r["subject_id"] for r in part} for name, part in cohorts.items()}
assert len({r["subject_id"] for r in rows}) == len(rows)
assert not (ids["development"] & ids["calibration"] or ids["development"] & ids["external"]
            or ids["calibration"] & ids["external"])
assert max(r["outcome_available_day"] for r in cohorts["development"]) < min(r["prediction_day"] for r in cohorts["calibration"])
assert max(r["outcome_available_day"] for r in cohorts["calibration"]) < min(r["prediction_day"] for r in cohorts["external"])
pprint({name: {"n": len(part), "events": sum(r["outcome"] for r in part),
               "statuses": dict(lab.Counter(lab.eligibility(r) for r in part))}
        for name, part in cohorts.items()})
pprint(rows[0])


## 2. Freeze calibration and the two policies

Fit only on site B. The five-bin smoothed mapping is a simple teaching estimator: empty bins use midpoint, and the mapping need not be monotone or improve external results. Site-B mapped performance is calibration fit, not independent evidence. Thresholds 0.25/0.60 are fixed teaching policies, not clinical choices.


In [ ]:
mapping = lab.fit_calibrator(cohorts["calibration"])
assert set(mapping["fit_subjects"]) <= ids["calibration"]
assert not set(mapping["fit_subjects"]) & ids["external"]
pprint(mapping)
external = cohorts["external"]
dev_eligible = [r for r in cohorts["development"] if lab.eligibility(r) == "eligible"]
assert dev_eligible, "No eligible development records for the baseline."
baseline_value = sum(r["outcome"] for r in dev_eligible)/len(dev_eligible)
baseline_rows = [dict(r, score=baseline_value if r["score"] is not None else None) for r in external]
reports = {}
for name, threshold in lab.POLICIES.items():
    reports[name] = {"original": lab.evaluate(external, threshold),
                     "mapped": lab.evaluate(external, threshold, mapping),
                     "constant_baseline": lab.evaluate(baseline_rows, threshold)}
pprint(reports)


## 3. Inspect calibration and probability error

Brier is mean squared probability error, not a pure calibration statistic. Empty bins stay missing. Point sizes show bin counts, not confidence. Average precision in reports is grouped-tie, non-interpolated AP, not trapezoidal PR-AUC. Review sparse bins and the constant baseline before drawing conclusions.


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 5))
for label, report in [("original", reports["sensitive"]["original"]),
                      ("mapped", reports["sensitive"]["mapped"])]:
    bins = [b for b in report["calibration"] if b["n"]]
    ax.scatter([b["mean_prediction"] for b in bins], [b["event_fraction"] for b in bins],
               s=[20+2*b["n"] for b in bins], alpha=.7, label=label)
    for b in bins:
        ax.annotate(str(b["n"]), (b["mean_prediction"], b["event_fraction"]), xytext=(4,4), textcoords="offset points")
ax.plot([0,1],[0,1], "--", color="gray")
ax.set(xlabel="Mean estimate in bin", ylabel="Observed event fraction", xlim=(0,1), ylim=(0,1),
       title="External synthetic cohort: bin counts annotated")
ax.legend(); plt.show()
if reports["sensitive"]["mapped"]["n_eligible"] >= 2:
    interval = lab.bootstrap_brier(external, mapping)
    pprint(interval)
else:
    print("Insufficient eligible records for the specified bootstrap.")


## 4. Pair performance with coverage

Compare G1/G2 total and eligible counts, rejection reasons, and conditional performance. An abstention is not a negative prediction. Descriptive site comparisons do not create multiple independent external sites: site B fitted the mapping, while only C is the external fixture. Explain that limit rather than estimating between-site generalization from this setup.


In [ ]:
group_reports = {group: lab.evaluate([r for r in external if r["group"] == group],
                                     lab.POLICIES["sensitive"], mapping)
                 for group in ("G1", "G2")}
pprint(group_reports)
for name, report in group_reports.items():
    assert report["tp"]+report["fp"]+report["fn"]+report["tn"] == report["n_eligible"]
    assert sum(report["status_counts"].values()) == report["n_total"]
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(list(group_reports), [v["coverage"] for v in group_reports.values()], color=["#167D91", "#7556A8"])
ax.set(ylim=(0,1), ylabel="Eligible / total", title="Coverage is part of the result")
plt.show()


## 5. Separate prevalence algebra from actual transport

Hold sensitivity 0.90 and specificity 0.80 fixed by assumption and vary prevalence. This is a constructed scenario; the generator's site change also alters the conditional relationship. The likelihood-ratio calculation below uses these declared inputs, not fitted clinical evidence.


In [ ]:
pprint([lab.prevalence_scenario(.90, .80, prevalence) for prevalence in (.02,.10,.30)])
lr_positive = .90/(1-.80)
prior = .10
post_odds = prior/(1-prior)*lr_positive
print({"LR_positive": lr_positive, "post_probability": post_odds/(1+post_odds)})


## 6. Compare modeled decision tradeoffs

The plot uses the same eligible external population for all strategies. Thresholds are a prespecified exploration grid, not a search for the best clinical cutoff. Net benefit assumes a threshold-based false-positive weight; the score-cutoff rule must be interpreted under its probability and decision assumptions. This is not a treatment-effect estimate, and it omits the consequences of handling ineligible people.


In [ ]:
grid = [i/20 for i in range(1,20)]
curves = [lab.evaluate(external, threshold, mapping) for threshold in grid]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(grid, [r["net_benefit_eligible"] for r in curves], label="mapped-score policy")
ax.plot(grid, [r["all_positive_eligible"] for r in curves], label="all positive")
ax.plot(grid, [r["all_negative_eligible"] for r in curves], label="all negative")
ax.set(xlabel="Threshold probability / implemented cutoff", ylabel="Net benefit among eligible records",
       title="Synthetic tradeoff analysis; no measured clinical impact")
ax.legend(); ax.grid(alpha=.2); plt.show()


## 7. Inspect research cards and export evidence

Outcome data belong to later evaluation, not the prediction-time card. The card's estimate and category are research teaching outputs with no patient-care action. The bundle includes evaluation rows separately, preserves the mapping and policies, and saves a byte digest. The digest is not authentication or immutability.


In [ ]:
eligible_row = next((r for r in external if lab.eligibility(r) == "eligible"), None)
abstain_row = next((r for r in external if lab.eligibility(r) != "eligible"), None)
for row in (eligible_row, abstain_row):
    if row is not None:
        card = lab.result_card(row, "sensitive", mapping)
        assert "outcome" not in card and "outcome_available_day" not in card
        pprint(card)
destination = lab.save_bundle(rows, mapping)
print(destination)


## 8. Write the translational report

Use `ASSIGNMENT.md`. Explain the supported claims and remaining measurement, reference, independent-validation, human-factors, and impact evidence. Report your own numerical results and failures. If an external result motivates a change, describe it as development and propose another independent evaluation. No result in this synthetic notebook establishes clinical readiness.
